In [6]:
# Sanity check — confirm v2 loader and database connectivity
import os, sys
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
load_dotenv(PROJECT_ROOT / ".env")

loader_src = (PROJECT_ROOT / "notebooks" / "loader.py").read_text()
v2_check = "PERFORMANCE INDEXES" in loader_src
print(f"loader.py is v2 (has indexes): {v2_check}")

driver = GraphDatabase.driver(os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD")))
with driver.session(database="system") as s:
    print("Databases available:")
    for r in s.run("SHOW DATABASES"):
        print(f"  {r['name']:10s} {r['currentStatus']}")
driver.close()
print("\nReady to run the loader.")

loader.py is v2 (has indexes): True
Databases available:
  hvac       online
  neo4j      online
  system     online

Ready to run the loader.


In [7]:
"""
M7U4 HVAC load — NBU MedicalClinic HVAC (TIB DURAARK dataset)
Target database: 'hvac' (Duplex graph stays in 'neo4j' untouched)
"""

import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# --- Paths ---
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
IFC_PATH = PROJECT_ROOT / "ifc" / "NBU_MedicalClinic_Eng-HVAC.ifc"

# --- Credentials ---
load_dotenv(PROJECT_ROOT / ".env")
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

# --- Sanity checks (must all pass before running the load) ---
assert IFC_PATH.exists(), f"IFC not found at {IFC_PATH}"
print(f"IFC file:  {IFC_PATH.name}  ({IFC_PATH.stat().st_size / 1024**2:.1f} MB)")

loader_src = (PROJECT_ROOT / "notebooks" / "loader.py").read_text()
assert "PERFORMANCE INDEXES" in loader_src, "loader.py is v1 — needs v2 with indexes"
print(f"loader.py: v2 (performance indexes present)")

driver = GraphDatabase.driver(URI, auth=(USER, PWD))
with driver.session(database="system") as s:
    dbs = {r["name"]: r["currentStatus"] for r in s.run("SHOW DATABASES")}
driver.close()
assert dbs.get("hvac") == "online", f"hvac database not online: {dbs}"
assert dbs.get("neo4j") == "online", f"neo4j database not online: {dbs}"
print(f"Databases: hvac={dbs['hvac']}, neo4j={dbs['neo4j']}")

print("\nAll sanity checks passed. Ready to load.")

IFC file:  NBU_MedicalClinic_Eng-HVAC.ifc  (26.1 MB)
loader.py: v2 (performance indexes present)
Databases: hvac=online, neo4j=online

All sanity checks passed. Ready to load.


In [8]:
from loader import IFCGraphLoader

loader = IFCGraphLoader(
    ifc_path=IFC_PATH,
    neo4j_uri=URI,
    neo4j_user=USER,
    neo4j_password=PWD,
    database="hvac",
    batch_size=500,
)

Opening IFC: D:\Projects\m7u4-ifc-graph\ifc\NBU_MedicalClinic_Eng-HVAC.ifc
  Schema: IFC2X3
  Elements: 3704
Target database: hvac


In [9]:
loader.run_all()


[1/6] Wipe and constrain
  done (15.8s)

[2/6] Spatial structure
  spatial nodes: {'Project': 1, 'Site': 1, 'Building': 1, 'Storey': 4, 'Space': 263}
  done (13.6s)

[3/6] Elements
    spatial_containment: 500/3704
    spatial_containment: 1000/3704
    spatial_containment: 1500/3704
    spatial_containment: 2000/3704
    spatial_containment: 2500/3704
    spatial_containment: 3000/3704
    spatial_containment: 3500/3704
    spatial_containment: 3704/3704
  loaded 3704 elements across 6 classes
  done (61.2s)

[4/6] Property sets and properties
    psets: 6500/33053
    psets: 13000/33053
    psets: 19500/33053
    psets: 26000/33053
    psets: 32500/33053
    psets: 33053/33053
    has_pset: 6500/33053
    has_pset: 13000/33053
    has_pset: 19500/33053
    has_pset: 26000/33053
    has_pset: 32500/33053
    has_pset: 33053/33053
    properties: 29500/148447
    properties: 59000/148447
    properties: 88500/148447
    properties: 118000/148447
    properties: 147500/148447
    prope

In [10]:
print("=" * 60)
print("Cross-database verification")
print("=" * 60)

verify_driver = GraphDatabase.driver(URI, auth=(USER, PWD))
for db in ["neo4j", "hvac"]:
    with verify_driver.session(database=db) as s:
        n = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
        r = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
        print(f"  {db:8s}  {n:>8} nodes  {r:>8} relationships")
verify_driver.close()

loader.close()
print("\nDone. Drivers closed.")

Cross-database verification
  neo4j        16178 nodes     16102 relationships
  hvac        193689 nodes    200262 relationships

Done. Drivers closed.
